# SportsAction: Video Action Recognition Framework

This notebook demonstrates how to use the SportsAction framework for video action recognition.

**Components:**
- **Spatial Branch**: CNN backbones (ResNet18, DenseNet121, Inception-v3)
- **Temporal Branch**: BiLSTM, LSTM, GRU, or TCN
- **Dataset**: PyTorch Dataset loader for video frames
- **Metrics**: F1-score, accuracy, precision, recall, IoU

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np

# Add project root to path
PROJECT_ROOT = Path.cwd()  # Adjust if running from different location
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_prep.dataset import SportsDataset
from src.models.video_classifier import VideoClassifier
from src.training.metrics import MetricsTracker

print(f"Project root: {PROJECT_ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch version: {torch.__version__}")

## 2. Load Dataset

In [ ]:
# Define paths
frames_dir = PROJECT_ROOT / "data" / "frames"

# Load datasets
train_dataset = SportsDataset(
    data_root=str(frames_dir),
    split="train",
    num_frames=16,
    backbone="resnet18"
)

val_dataset = SportsDataset(
    data_root=str(frames_dir),
    split="val",
    num_frames=16,
    backbone="resnet18"
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Classes: {train_dataset.classes}")
print(f"Num classes: {len(train_dataset.classes)}")

## 3. Create DataLoaders

In [ ]:
def collate_skip_none(batch):
    """Skip any None items in a batch."""
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    return torch.utils.data.dataloader.default_collate(batch)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,  # Set to 0 in notebook, >0 for scripts
    collate_fn=collate_skip_none
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_skip_none
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 4. Inspect a Batch

In [ ]:
# Get a sample batch
sample_batch = next(iter(train_loader))

if sample_batch is not None:
    clips, labels = sample_batch
    print(f"Batch shape: {clips.shape}")
    print(f"  - Batch size: {clips.shape[0]}")
    print(f"  - Num frames: {clips.shape[1]}")
    print(f"  - Channels: {clips.shape[2]}")
    print(f"  - Height: {clips.shape[3]}")
    print(f"  - Width: {clips.shape[4]}")
    print(f"\nLabels: {labels}")
    print(f"Label range: [{labels.min().item()}, {labels.max().item()}]")
else:
    print("Batch is None - check that frames are extracted properly")

## 5. Create Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model configuration
num_classes = len(train_dataset.classes)

model = VideoClassifier(
    backbone_name="resnet18",
    num_classes=num_classes,
    temporal="bilstm",
    pretrained_backbone=True,
    lstm_hidden=256,
    lstm_layers=1,
    bidirectional=True
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: ResNet18 + BiLSTM")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Device: {device}")

## 6. Forward Pass Test

In [ ]:
# Test forward pass
model.eval()

with torch.no_grad():
    sample_batch = next(iter(train_loader))
    if sample_batch is not None:
        clips, labels = sample_batch
        clips = clips.to(device)
        
        logits = model(clips)
        predictions = torch.argmax(logits, dim=1)
        probabilities = torch.softmax(logits, dim=1)
        
        print(f"Logits shape: {logits.shape}")
        print(f"Predictions: {predictions}")
        print(f"Ground truth: {labels}")
        print(f"\nProbabilities (first sample): {probabilities[0]}")
        print(f"Confidence (first sample): {probabilities[0].max().item():.4f}")
    else:
        print("Failed to load batch")

## 7. Compute Metrics

In [ ]:
# Validate on all validation data
model.eval()
metrics_tracker = MetricsTracker()

with torch.no_grad():
    for batch in val_loader:
        if batch is None:
            continue
        
        clips, labels = batch
        clips = clips.to(device)
        
        logits = model(clips)
        predictions = torch.argmax(logits, dim=1)
        
        metrics_tracker.update(predictions, labels)

# Get metrics
metrics = metrics_tracker.get_metrics()

print("Validation Metrics:")
print(f"  Accuracy: {metrics.get('accuracy', 0):.4f}")
print(f"  F1-Score: {metrics.get('f1', 0):.4f}")
print(f"  Precision: {metrics.get('precision', 0):.4f}")
print(f"  Recall: {metrics.get('recall', 0):.4f}")

## 8. Compare Different Model Variants

In [ ]:
# Test different architecture combinations
configs = [
    {"backbone": "resnet18", "temporal": "bilstm"},
    {"backbone": "resnet18", "temporal": "none"},
    {"backbone": "resnet18", "temporal": "lstm"},
    {"backbone": "densenet121", "temporal": "bilstm"},
]

for config in configs:
    model_test = VideoClassifier(
        backbone_name=config["backbone"],
        num_classes=num_classes,
        temporal=config["temporal"],
        pretrained_backbone=False,
    )
    
    params = sum(p.numel() for p in model_test.parameters())
    print(f"{config['backbone']:15s} + {config['temporal']:8s}: {params:,} params")

## Next Steps

1. **Extract Frames**: Run `src/data_prep/extract_frames.py` to extract frames from videos
2. **Train Model**: Use `src/training/train.py` for full training with logging
3. **Evaluate**: Run `src/training/test.py` to evaluate on test set
4. **Visualize**: Use `src/utils/visualizer.py` to overlay predictions on video frames

See [README.md](../README.md) for more details.